# setup

> The first run: ramabana starts inside its own tmux server, offers to install tmux when it is missing, and `ramabana --doctor` says what it found.

In [ ]:
#| default_exp setup

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq

In [ ]:
#| export
from __future__ import annotations

import json, os, platform, re, shlex, shutil, subprocess, sys, uuid

from fastcore.all import Path, ifnone, patch, store_attr

from ramabana import core

## The config

The now pane needs a tmux split, and shift+enter needs tmux to pass extended keys through. Each session gets a tmux server of its own, `tmux -L ramabana-<id>`, started with `<cfg>/tmux.conf`. A fresh server reads the current config and inherits the current environment, so a second session never picks up the first one's API keys or virtualenv. That server never reads `~/.tmux.conf`, and your own tmux is left as it was.

`tmux_conf` adds only the lines the installed tmux understands. `extended-keys` arrived in tmux 3.2 and `extended-keys-format csi-u` in 3.5. An older tmux would print an error for each unknown line when the session opens. `RGB` is claimed only when `$COLORTERM` says the terminal draws true colour.

In [ ]:
#| export
SOCKET, WRAP_MIN, SPLIT_MIN, KEYS_MIN = 'ramabana', (3, 0), (3, 1), (3, 2)
TMUX_CONF = r'''# ramabana writes this file and rewrites it when its own copy changes
if-shell 'infocmp tmux-256color' 'set -g default-terminal tmux-256color' 'set -g default-terminal screen-256color'
if-shell '[ "$COLORTERM" = truecolor ] || [ "$COLORTERM" = 24bit ]' "set -as terminal-overrides ',*:RGB'"
set -s escape-time 10
set -g mouse on
set -g status off
set -g remain-on-exit off
set -g destroy-unattached on
'''
VERSIONED = {KEYS_MIN: "set -s extended-keys on\nset -as terminal-features ',*:extkeys'",
             (3, 5): 'set -s extended-keys-format csi-u'}

def tmux_version(s):
    "The `(major, minor)` in `tmux -V` output such as `tmux 3.5a`, or None."
    return tuple(map(int, m.groups())) if (m := re.search(r'(\d+)\.(\d+)', s or '')) else None

def tmux_conf(version):
    "`TMUX_CONF` with the lines tmux `version` understands."
    return TMUX_CONF + ''.join(f'{l}\n' for v, l in VERSIONED.items() if version >= v)

In [ ]:
test_eq([tmux_version(s) for s in ('tmux 3.5a', 'tmux next-3.6', 'tmux master')], [(3, 5), (3, 6), None])
assert 'extended-keys-format csi-u' in tmux_conf((3, 5)) and 'csi-u' not in tmux_conf((3, 4))
assert 'extended-keys on' in tmux_conf((3, 2)) and 'extended-keys' not in tmux_conf((3, 1))
print(tmux_conf((3, 5)))

## The relaunch

`Setup.launch` runs before an interactive session starts. Outside tmux it replaces this process with `tmux new-session` on a new server, running the same command line in the same folder. Quitting ramabana ends the session and returns you to the shell. It stays put for a one-shot question, for `--json`, when stdin or stdout is not a terminal, and when `--tmux off` or `$RAMABANA_TMUX=off` says to. The relaunched process has `$RAMABANA_WRAPPED` set, so it never relaunches again. Inside tmux, `--pane auto` then opens the pane.

`destroy-unattached` ends the session when you detach or close the window, so ramabana never runs on unseen. The history is saved, and `--resume latest` reopens it. The command runs under `sh`, which waits for enter after a non-zero exit, so an error stays on screen instead of vanishing with the session.

Inside a tmux of your own, `launch` turns `extended-keys` on for that server when it is off. The change lasts until the server exits, and only programs that ask for extended keys get them.

Every outside effect goes through a seam: `which`, `system`, `run`, `execvpe`, `ask` and `isatty`, plus `environ`, `argv` and `cwd`.

In [ ]:
#| export
HOLD = r'"$@" || { s=$?; printf "\nramabana exited with status %s · enter closes this\n" $s; read _; exit $s; }'
WHY = {'none': 'no tmux, and no brew, apt-get or dnf to install it', 'no': 'no tmux', '': 'no tmux', 'failed': 'tmux did not install'}

class Setup:
    "The tmux ramabana runs in: its config, the relaunch into it, the install offer, and `--doctor`."
    def __init__(self,
        cfg='~/.config/ramabana', # holds `tmux.conf` and `setup.json`
        environ=None,             # default `os.environ`
        argv=None,                # the command line to rerun; default `sys.argv`
        cwd=None,                 # where the session starts; default the current folder
        which=shutil.which, system=platform.system, run=subprocess.run, execvpe=os.execvpe, ask=input,
        isatty=None,              # whether stdin and stdout are terminals
    ):
        store_attr('which,system,run,execvpe,ask')
        self._about = None
        self.cfg, self.environ, self.argv = Path(cfg or '~/.config/ramabana').expanduser(), ifnone(environ, os.environ), ifnone(argv, sys.argv)
        self.cwd, self.isatty = cwd or os.getcwd(), isatty or (lambda: sys.stdin.isatty() and sys.stdout.isatty())

    def _env(self, name): return self.environ.get(core.ENV_PREFIX+name) or self.environ.get(core.ENV_FALLBACK+name)
    @property
    def tmux(self): return self.which('tmux')

    def about(self):
        "`tmux -V`, asked once, or '' without tmux."
        if self._about is None and (t := self.tmux): self._about = self.run([t, '-V'], capture_output=True, text=True).stdout.strip()
        return self._about or ''

    def version(self):
        "The `(major, minor)` of tmux, `(99, 0)` for an unnumbered build, or None without tmux."
        return (tmux_version(a) or (99, 0)) if (a := self.about()) else None

    def conf(self):
        "Write `tmux_conf` for this tmux to `<cfg>/tmux.conf` when it differs, and return the path."
        p, text = self.cfg/'tmux.conf', tmux_conf(self.version() or (0, 0))
        if not p.exists() or p.read_text() != text: p.mk_write(text)
        return p

    def command(self):
        "The argv that reruns this command line: its script, else this Python."
        exe, *args = self.argv
        me = [exe] if os.path.isfile(exe) and os.access(exe, os.X_OK) else [sys.executable, '-c', 'import sys; from ramabana.cli import main; sys.exit(main())']
        return [*me, *args]

    def wrap(self):
        "The `execvpe` arguments that rerun `command` on a tmux server of its own."
        env = {**self.environ, core.ENV_PREFIX+'WRAPPED': '1'}
        return self.tmux, ['tmux', '-L', f'{SOCKET}-{uuid.uuid4().hex[:8]}', '-f', str(self.conf()), 'new-session', '-s', SOCKET,
                           '-c', self.cwd, 'sh', '-c', HOLD, 'ramabana', *self.command()], env

    def launch(self,
        prompt='',     # a one-shot question stays in this terminal
        as_json=False, # so does `--json`
        tmux='auto',   # `--tmux`; `off` stays out of tmux
    ):
        "Before an interactive session: ready tmux, and rerun inside ramabana's own server when outside tmux."
        if prompt or as_json or not self.isatty() or 'off' in (tmux, self._env('TMUX')): return
        if self.environ.get('TMUX'): return self.ext_keys()
        if self._env('WRAPPED') or not (self.tmux or self.offer_install()) or (self.version() or (0,)) < WRAP_MIN: return
        self.execvpe(*self.wrap())

A fake machine shows the relaunch. It has tmux 3.5 on `PATH`, and `execvpe` records what it would run:

In [ ]:
from types import SimpleNamespace
import tempfile

class Box:
    "Programs on PATH, the commands run, the prompts shown and the exec made."
    def __init__(self, have=('tmux',), version='tmux 3.5a', system='Darwin', answer='', ext='on'):
        self.have, self.version, self.system, self.answer, self.ext = set(have), version, system, answer, ext
        self.calls, self.asked, self.execd = [], [], None
    def which(self, name): return f'/bin/{name}' if name in self.have else None
    def run(self, cmd, **kw):
        self.calls.append(list(cmd))
        if 'install' in cmd: self.have.add('tmux')
        return SimpleNamespace(returncode=0, stdout={'-V': self.version, 'show': self.ext}.get(cmd[1], ''))
    def ask(self, q): return self.asked.append(q) or self.answer
    def execvpe(self, *a): self.execd = a

def mk(box, **kw):
    return Setup(tempfile.mkdtemp(), environ=kw.pop('environ', {'PATH': '/bin'}), argv=['-c', '--pane', 'on'], cwd='/work',
                 which=box.which, system=lambda: box.system, run=box.run, execvpe=box.execvpe, ask=box.ask, isatty=lambda: True)

box = Box()
mk(box).launch()
exe, argv, env = box.execd
test_eq((exe, env['RAMABANA_WRAPPED'], argv[argv.index('-c'):][:2], argv[-2:]), ('/bin/tmux', '1', ['-c', '/work'], ['--pane', 'on']))
argv[:12]

A one-shot question, `--json` and `--tmux off` stay in this terminal:

In [ ]:
for kw in (dict(prompt='hi'), dict(as_json=True), dict(tmux='off')):
    box = Box()
    mk(box).launch(**kw)
    test_eq(box.execd, None)

## Installing tmux

When tmux is missing, `offer_install` asks once whether to run the command that installs it, and names it: `brew install tmux` on macOS, `sudo apt-get install -y tmux` or `sudo dnf install -y tmux` on Linux. The command runs attached to the terminal, so `sudo` can ask for a password. A no, or a yes whose install worked, goes in `<cfg>/setup.json`, and a no is never asked again. Ctrl+c at the prompt, or a failed or interrupted install, saves nothing, so the next run asks again. `ramabana --doctor` asks even so. Without tmux, the session starts in this terminal and one line says how to get the pane.

In [ ]:
#| export
@patch
def installer(self:Setup):
    "The command that installs tmux here, or None."
    if self.system() == 'Darwin': return ['brew', 'install', 'tmux'] if self.which('brew') else None
    pm = next((p for p in ('apt-get', 'dnf') if self.which(p)), None) if self.system() == 'Linux' else None
    return pm and [*(['sudo'] if self.which('sudo') else []), pm, 'install', '-y', 'tmux']

@patch
def _seen(self:Setup):
    try: return (self.cfg/'setup.json').read_json()
    except (OSError, ValueError): return {}

@patch
def _answer(self:Setup, cmd):
    "'yes' or 'no' to the install, or '' when ctrl+c or end of input left it unanswered."
    try: yes = self.ask(f'ramabana opens its now pane in tmux, which is missing. Run `{shlex.join(cmd)}`? [y/N] ').strip().lower() in ('y', 'yes')
    except (EOFError, KeyboardInterrupt): return ''
    return 'yes' if yes else 'no'

@patch
def _installed(self:Setup, cmd):
    try: return self.run(cmd).returncode == 0 and bool(self.tmux)
    except (OSError, KeyboardInterrupt): return False

@patch
def offer_install(self:Setup, force=False):
    "Ask once, or again when `force`, to install tmux: True when tmux is there now."
    if not force and 'install_tmux' in self._seen(): return False
    ans = 'none' if (cmd := self.installer()) is None else self._answer(cmd)
    if ans == 'yes' and not self._installed(cmd): ans = 'failed'
    if ans in ('none', 'no', 'yes'): (self.cfg/'setup.json').mk_write(json.dumps(self._seen() | {'install_tmux': ans}))
    if ans == 'yes': return True
    print(f'{WHY[ans]}; continuing without tmux · ramabana --doctor shows how to get the now pane', file=sys.stderr)
    return False

In [ ]:
box = Box(have={'brew'}, answer='y')
mk(box).launch()
test_eq((box.calls[0], box.execd is not None), (['brew', 'install', 'tmux'], True))
box.asked

In [ ]:
box = Box(have={'apt-get', 'sudo'}, system='Linux', answer='n')
s = mk(box)
s.launch()
s.launch()
test_eq((len(box.asked), box.execd, s._seen()), (1, None, {'install_tmux': 'no'}))

## Your own tmux

`ext_keys` reads `extended-keys` from the server it runs inside and turns it on only when it is `off`. Your `~/.tmux.conf` is never touched.

In [ ]:
#| export
@patch
def _show_ext(self:Setup):
    "This tmux server's `extended-keys`, or '' when tmux cannot say."
    if not (t := self.tmux): return ''
    r = self.run([t, 'show', '-sv', 'extended-keys'], capture_output=True, text=True)
    return '' if r.returncode else r.stdout.strip()

@patch
def ext_keys(self:Setup):
    "Turn on `extended-keys` in the tmux server we run inside when it is off: True if turned on."
    if self._show_ext() != 'off': return False
    self.run([self.tmux, 'set', '-s', 'extended-keys', 'on'], capture_output=True)
    print('turned on extended-keys in this tmux server so shift+enter reaches ramabana; ~/.tmux.conf is unchanged', file=sys.stderr)
    return True

In [ ]:
for ext, sets in (('off', True), ('on', False), ('always', False)):
    box = Box(ext=ext)
    mk(box, environ={'TMUX': '/tmp/tmux-501/default,1,0'}).launch()
    test_eq((['/bin/tmux', 'set', '-s', 'extended-keys', 'on'] in box.calls, box.execd), (sets, None))

## `--doctor`

`ramabana --doctor` prints one line for each thing the pane and the keys depend on, then offers the install when tmux is missing, even after an earlier no.

In [ ]:
#| export
@patch
def doctor(self:Setup):
    "Print one line per check, offer to install tmux when it is missing, and return 0."
    v, about, conf, pane = self.version(), self.about(), self.cfg/'tmux.conf', self.which('ramabana-pane')
    keys = f'{self._show_ext() or "unknown"} in this tmux' if self.environ.get('TMUX') else \
           f"{'on' if v and v >= KEYS_MIN else 'needs tmux 3.2+'} in ramabana's own tmux server"
    split = f'ok on {about}' if v and v >= SPLIT_MIN else 'needs tmux 3.1+' + (f', this is {about}; /pane prints a command to run instead' if v else '')
    print(f'tmux: {about} at {self.tmux}' if v else 'tmux: not found', f'split: {split}', f'extended-keys: {keys}',
          f'config: {conf}' + ('' if conf.exists() else ', written by the first session'),
          f'ramabana-pane: {pane}' if pane else 'ramabana-pane: not on PATH; the pane runs python -m ramabana.pane', sep='\n')
    if not v: self.offer_install(force=True)
    return 0

In [ ]:
mk(Box(have={'tmux', 'ramabana-pane'}, version='tmux 3.0a')).doctor()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()